# Week 7 Homework: Neural Networks and Algorithmic Fairness

## Purpose of Homework

This homework will give you practice building and comparing three types of predictive models — **logistic regression**, **decision trees**, and **neural networks** — applied to a high-stakes real-world problem: predicting criminal recidivism. You will use data from the COMPAS (Correctional Offender Management Profiling for Alternative Sanctions) algorithm, which has been used by courts across the United States to inform bail, sentencing, and parole decisions.

You will also examine the **ethical implications** of using algorithmic tools in criminal justice, with particular attention to fairness and racial bias.

You are encouraged to refer to lecture content and liberally use course resources such as the discussion board and office hours.

## Required Reading

**Before starting this homework, please read the following:**

1. **ProPublica article**: [Machine Bias](https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing) — Julia Angwin, Jeff Larson, Surya Mattu, and Lauren Kirchner (2016). This investigative journalism piece examines COMPAS scores and their racial disparities.

2. **ProPublica methodology**: [How We Analyzed the COMPAS Recidivism Algorithm](https://www.propublica.org/article/how-we-analyzed-the-compas-recidivism-algorithm) — explains the statistical approach used to evaluate the algorithm.

Both links are also available on the [ProPublica COMPAS Analysis GitHub repository](https://github.com/propublica/compas-analysis).

## Logistics

Due date: The homework is due **11:59pm on Thursday, March 19, 2026**.

You will submit your homework on [MarkUs](https://markus.teach.cs.toronto.edu/markus/).

1. Download this file (`STA272_hw7_student.ipynb`) from JupyterHub. (See [our JupyterHub Guide](../guides/jupyterhub_guide.ipynb) for detailed instructions.)
2. Submit this file to MarkUs under the hw7 assignment. (See [our MarkUs Guide](../guides/markus_guide.ipynb) for detailed instructions.)

All homeworks will take place in a Jupyter notebook (like this one). When you are done, you will download this notebook and submit it to MarkUs.

## About the Data

The **COMPAS** algorithm assigns a **recidivism risk score** (1–10) to criminal defendants to predict whether they will re-offend within two years. These scores are used by judges and parole boards in many US jurisdictions.

ProPublica obtained COMPAS scores for defendants in Broward County, Florida, and linked them to actual criminal records to check the algorithm's accuracy and fairness. Their analysis revealed that COMPAS scores disproportionately labelled Black defendants as higher-risk compared to White defendants with similar criminal histories.

**Research Question:** Can we build models that predict two-year recidivism using demographic and criminal history features? How do they compare to COMPAS, and are they fair?

### Target Variable

| Variable | Description | Type |
|----------|-------------|------|
| `two_year_recid` | Re-arrested within two years (1=Yes, 0=No) | Binary |

### Predictor Variables

| Variable | Description | Type |
|----------|-------------|------|
| `age` | Age at time of screening | Numeric |
| `sex` | Sex | Categorical (Male/Female) |
| `race` | Race | Categorical (6 groups) |
| `priors_count` | Number of prior offenses | Numeric |
| `juv_fel_count` | Number of juvenile felonies | Numeric |
| `juv_misd_count` | Number of juvenile misdemeanours | Numeric |
| `juv_other_count` | Number of other juvenile offenses | Numeric |
| `c_charge_degree` | Current charge: Felony (F) or Misdemeanour (M) | Categorical |

### Additional Columns (for context, not used as predictors)

| Variable | Description |
|----------|-------------|
| `decile_score` | COMPAS risk score (1=Low, 10=High) |
| `score_text` | COMPAS risk category (Low/Medium/High) |

## Task #1: Load and Filter the COMPAS Data

Download `compas-scores-two-years.csv` from the [ProPublica COMPAS analysis repository](https://github.com/propublica/compas-analysis) and place it in the same folder as this notebook.

Load the data into a DataFrame called `compas_df`. Then, following [ProPublica's filtering methodology](https://www.propublica.org/article/how-we-analyzed-the-compas-recidivism-algorithm), apply the following five filters and store the result in `compas_filtered`:

1. `days_b_screening_arrest >= -30` (screening within 30 days of arrest)
2. `days_b_screening_arrest <= 30`
3. `is_recid != -1` (exclude cases with unknown recidivism status)
4. `c_charge_degree != 'O'` (exclude "other" charges, mostly traffic offenses)
5. `score_text != 'N/A'` (exclude cases with missing COMPAS scores)

Print the shape of `compas_filtered` and the overall two-year recidivism rate.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #1 in this cell

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the data
compas_df = pd.read_csv('compas-scores-two-years.csv')

print(f'Raw data shape: {compas_df.shape}')

# Apply ProPublica filters — fill in the ...
compas_filtered = compas_df[
    (compas_df['days_b_screening_arrest'] >= ...) &
    (compas_df['days_b_screening_arrest'] <= ...) &
    (compas_df['is_recid'] != ...) &
    (compas_df['c_charge_degree'] != ...) &
    (compas_df['score_text'] != ...)
].copy()

print(f'Filtered data shape: {compas_filtered.shape}')
print(f'Two-year recidivism rate: {compas_filtered["two_year_recid"].mean():.1%}')

## Task #2: Explore the Data

Using the filtered data from Task #1:

1. Compute the **mean COMPAS decile score by race** using `groupby` and store the result as `mean_score_by_race` (a Series).
2. Print `mean_score_by_race`.
3. Create **two side-by-side plots**:
   - **Left**: A histogram of `decile_score` (the COMPAS risk score).
   - **Right**: A horizontal bar chart showing the actual two-year recidivism rate by race.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #2 in this cell

# Compute mean COMPAS decile score by race
mean_score_by_race = compas_filtered.groupby(...)['decile_score'].mean()
print('Mean COMPAS decile score by race:')
print(mean_score_by_race.sort_values(ascending=False).round(2))

# Side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)

compas_filtered['decile_score'].hist(ax=axes[0], bins=10, color='steelblue', edgecolor='white')
axes[0].set_xlabel('COMPAS Decile Score (1=Low Risk, 10=High Risk)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of COMPAS Decile Scores')

race_recid = (compas_filtered.groupby('race')['two_year_recid']
              .mean().sort_values(ascending=True))
race_recid.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_xlabel('Two-Year Recidivism Rate')
axes[1].set_title('Actual Recidivism Rate by Race')
plt.tight_layout()
plt.show()

## Task #3: Feature Engineering

Prepare the features for modelling:

1. Clean the `race` column: replace hyphens and spaces with underscores and store the result in a new column called `race_clean`.
2. Create a binary `sex_female` column (1 = Female, 0 = Male).
3. Create a binary `charge_felony` column (1 = Felony, 0 = Misdemeanour).
4. Apply `pd.get_dummies()` to encode `race_clean` with `drop_first=False`. Store the result in `compas_encoded`.
5. Define `feature_cols` as the list below and create `X = compas_encoded[feature_cols]` and `y = compas_encoded['two_year_recid']`.

```python
feature_cols = [
    'age', 'sex_female', 'priors_count',
    'juv_fel_count', 'juv_misd_count', 'juv_other_count',
    'charge_felony',
    'race_clean_African_American', 'race_clean_Asian', 'race_clean_Caucasian',
    'race_clean_Hispanic', 'race_clean_Native_American', 'race_clean_Other'
]
```

Print the shape of `X` and the class distribution of `y`.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #3 in this cell

# Step 1: Clean race column
compas_filtered['race_clean'] = compas_filtered['race'].str.replace('-', '_').str.replace(' ', '_')

# Step 2: Create binary sex variable (1=Female, 0=Male)
compas_filtered['sex_female'] = (compas_filtered['sex'] == ...).astype(int)

# Step 3: Create binary charge degree variable (1=Felony, 0=Misdemeanour)
compas_filtered['charge_felony'] = (compas_filtered['c_charge_degree'] == ...).astype(int)

# Step 4: One-hot encode race_clean
compas_encoded = pd.get_dummies(compas_filtered, columns=['race_clean'], drop_first=...)

# Step 5: Define feature columns and create X, y
feature_cols = [
    'age', 'sex_female', 'priors_count',
    'juv_fel_count', 'juv_misd_count', 'juv_other_count',
    'charge_felony',
    'race_clean_African_American', 'race_clean_Asian', 'race_clean_Caucasian',
    'race_clean_Hispanic', 'race_clean_Native_American', 'race_clean_Other'
]

X = compas_encoded[...]
y = compas_encoded[...]

print(f'X shape: {X.shape}')
print(f'y distribution:\n{y.value_counts()}')
print(f'Recidivism rate: {y.mean():.1%}')

## Task #4: Train/Test Split and Feature Scaling

Split the data into training and test sets using `train_test_split` with:
- Test size: 20%
- `random_state=272`

Store the results as `X_train`, `X_test`, `y_train`, `y_test`.

Then standardize the features using `StandardScaler`:
- Fit the scaler **only** on the training data and store as `scaler`.
- Transform the training features and store as `X_train_scaled`.
- Transform the test features (using the same fitted scaler) and store as `X_test_scaled`.

**Note:** Decision trees do not require feature scaling. Logistic regression and neural networks do.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #4 in this cell

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=..., random_state=...
)

print(f'Training set: {len(X_train):,} observations')
print(f'Test set:     {len(X_test):,} observations')
print(f'Recidivism rate (train): {y_train.mean():.1%}')
print(f'Recidivism rate (test):  {y_test.mean():.1%}')

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(...)
X_test_scaled = scaler.transform(...)

print(f'\nScaled features shape: {X_train_scaled.shape}')
print(f'Mean of first scaled column (should be ~0): {X_train_scaled[:, 0].mean():.6f}')

## Task #5: Logistic Regression (Baseline Model)

Fit a `LogisticRegression` model with `max_iter=1000` and `random_state=272` on the **scaled** training data. Store the fitted model as `lr_model`.

Compute and store:
- `lr_accuracy`: test set accuracy (proportion of correct predictions)
- `lr_auc`: test set AUC (area under the ROC curve) using `roc_auc_score`

Also print the full classification report (precision, recall, F1-score for each class).

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #5 in this cell

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Fit logistic regression on scaled training data
lr_model = LogisticRegression(max_iter=..., random_state=...)
lr_model.fit(..., ...)

# Predictions
lr_pred = lr_model.predict(...)
lr_prob = lr_model.predict_proba(...)[:, 1]  # probability of recidivism

# Metrics
lr_accuracy = accuracy_score(..., ...)
lr_auc = roc_auc_score(..., ...)

print(f'Logistic Regression Test Accuracy: {lr_accuracy:.4f}')
print(f'Logistic Regression Test AUC:      {lr_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, lr_pred, target_names=['No Recidivism', 'Recidivism']))

## Task #6: Decision Tree

We used 5-fold cross-validation on the training set to select the best `max_depth` for the decision tree. The best depth was **`best_depth = 4`**.

Fit a `DecisionTreeClassifier` with `max_depth=best_depth` and `random_state=272` on the **unscaled** training data (`X_train` — decision trees do not require scaling). Store as `dt_model`.

Compute and store `dt_accuracy` and `dt_auc` on the test set. Plot the fitted tree.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #6 in this cell

from sklearn.tree import DecisionTreeClassifier, plot_tree

# Best depth was selected via 5-fold cross-validation
best_depth = 4

dt_model = DecisionTreeClassifier(max_depth=..., random_state=272)
dt_model.fit(..., ...)

# Evaluate on test set
dt_pred = dt_model.predict(...)
dt_prob = dt_model.predict_proba(...)[:, 1]
dt_accuracy = accuracy_score(..., ...)
dt_auc = roc_auc_score(..., ...)

print(f'Decision Tree Test Accuracy: {dt_accuracy:.4f}')
print(f'Decision Tree Test AUC:      {dt_auc:.4f}')
print(f'Tree depth: {dt_model.get_depth()},  Leaves: {dt_model.get_n_leaves()}')

# Plot the tree (provided)
plt.figure(figsize=(20, 8), dpi=150)
plot_tree(dt_model, feature_names=feature_cols, class_names=['No Recidivism', 'Recidivism'],
          filled=True, rounded=True, fontsize=7, precision=2)
plt.title(f'Decision Tree (max_depth={best_depth}) — Test Accuracy: {dt_accuracy:.3f}  AUC: {dt_auc:.3f}')
plt.tight_layout()
plt.show()

## Task #7: Neural Network

**Step 1:** Try the following four network architectures. For each, fit an `MLPClassifier` with `max_iter=1000` and `random_state=272` on the **scaled** training data and compute test accuracy and AUC. Store the results in `arch_df`.

```python
architectures   = [(32,),  (64, 32),  (128, 64, 32),  (100, 100)]
arch_labels     = ['(32,)', '(64, 32)', '(128, 64, 32)', '(100, 100)']
```

Print `arch_df`.

**Step 2:** Choose `best_arch` as the architecture with the highest AUC from `arch_df`. Fit a final model with that architecture, `max_iter=1000`, and `random_state=272` on the scaled training data. Store as `nn_model`.

Compute and store `nn_accuracy` and `nn_auc` on the test set.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #7 in this cell

from sklearn.neural_network import MLPClassifier

architectures = [(32,), (64, 32), (128, 64, 32), (100, 100)]
arch_labels   = ['(32,)', '(64, 32)', '(128, 64, 32)', '(100, 100)']

# Step 1: Try each architecture
arch_results = []
for arch, label in zip(architectures, arch_labels):
    nn = MLPClassifier(hidden_layer_sizes=..., max_iter=..., random_state=...)
    nn.fit(..., ...)
    acc = accuracy_score(y_test, nn.predict(...))
    auc = roc_auc_score(y_test, nn.predict_proba(...)[:, 1])
    arch_results.append({'Architecture': label, 'Accuracy': round(acc, 4), 'AUC': round(auc, 4)})

arch_df = pd.DataFrame(arch_results)
print(arch_df.to_string(index=False))

# Step 2: Fit final model with the best architecture
best_arch = ...  # fill in based on arch_df (e.g., (64, 32))

nn_model = MLPClassifier(hidden_layer_sizes=..., max_iter=1000, random_state=272)
nn_model.fit(..., ...)

nn_pred = nn_model.predict(...)
nn_prob = nn_model.predict_proba(...)[:, 1]
nn_accuracy = accuracy_score(..., ...)
nn_auc = roc_auc_score(..., ...)

print(f'\nFinal Neural Network Test Accuracy: {nn_accuracy:.4f}')
print(f'Final Neural Network Test AUC:      {nn_auc:.4f}')

## Task #8: Model Comparison

Build a summary `comparison` DataFrame that compares all models — including the **COMPAS algorithm** as a benchmark.

The COMPAS algorithm flags defendants as high-risk when `decile_score >= 5`. We can compute its accuracy and AUC against actual recidivism on the test set.

The COMPAS comparison code is provided below. Your task is to fill in the `comparison` DataFrame.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #8 in this cell

# COMPAS benchmark (provided code)
decile_test = compas_encoded.loc[X_test.index, 'decile_score']
compas_pred = (decile_test >= 5).astype(int)
compas_accuracy = accuracy_score(y_test, compas_pred)
compas_auc = roc_auc_score(y_test, decile_test)

print(f'COMPAS Test Accuracy: {compas_accuracy:.4f}')
print(f'COMPAS Test AUC:      {compas_auc:.4f}')

# Build the comparison DataFrame — fill in the ...
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Neural Network', 'COMPAS'],
    'Accuracy': [round(lr_accuracy, 4), round(..., 4), round(..., 4), round(compas_accuracy, 4)],
    'AUC': [round(lr_auc, 4), round(..., 4), round(..., 4), round(compas_auc, 4)]
})

print('\nModel Comparison:')
print(comparison.to_string(index=False))

## Task #9: Fairness Analysis

A model may be accurate overall but systematically unfair to particular groups. One important fairness metric is the **false positive rate (FPR)**: the proportion of people who did *not* recidivate that were incorrectly predicted to be high-risk.

In criminal justice, a false positive means someone is flagged as high-risk when they would not have re-offended — potentially leading to harsher sentences or denied parole.

Compute the FPR for **African-American** and **Caucasian** defendants separately, for both your neural network and COMPAS. Build a DataFrame called `fairness_df` with columns `'Race'`, `'N'`, `'NN False Positive Rate'`, and `'COMPAS False Positive Rate'`.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #9 in this cell

# Assemble test set results: combine features, true labels, and predictions
test_results = X_test.copy()
test_results['actual']       = y_test.values
test_results['nn_pred']      = nn_pred
test_results['compas_pred']  = compas_pred.values

# We will compute the false positive rate (FPR) for each race group.
# FPR = P(predicted high-risk | actually did NOT recidivate)
#     = (# incorrectly flagged as high-risk) / (# who did not recidivate)
# We use the one-hot encoded race columns to identify each group.
race_groups = {
    'African-American': test_results['race_clean_African_American'] == 1,
    'Caucasian':        test_results['race_clean_Caucasian'] == 1
}

fairness_rows = []
# .items() loops over the dictionary's key-value pairs. On each iteration:
#   - `race` gets the key: the group name (e.g., 'African-American')
#   - `mask` gets the value: a boolean Series that is True for rows belonging to that race
for race, mask in race_groups.items():
    # Select only defendants of this race
    group = test_results[mask]

    # Filter to people who did NOT recidivate (actual == 0)
    # These are the people for whom a "high-risk" prediction is a false positive
    non_recid_group = group[group['actual'] == ...]

    # Neural network FPR: among non-recidivists, what fraction were predicted 1 (high-risk)?
    nn_fpr = (non_recid_group['nn_pred'] == ...).mean()

    # COMPAS FPR: same calculation using COMPAS predictions
    compas_fpr = (non_recid_group['compas_pred'] == ...).mean()

    fairness_rows.append({
        'Race': race,
        'N': int(mask.sum()),
        'NN False Positive Rate': round(nn_fpr, 4),
        'COMPAS False Positive Rate': round(compas_fpr, 4)
    })

fairness_df = pd.DataFrame(fairness_rows)
print(fairness_df.to_string(index=False))

# Bar chart comparison (provided)
x = np.arange(len(fairness_df))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
ax.bar(x - width/2, fairness_df['NN False Positive Rate'],   width, label='Neural Network', color='steelblue')
ax.bar(x + width/2, fairness_df['COMPAS False Positive Rate'], width, label='COMPAS',         color='coral')
ax.set_xticks(x)
ax.set_xticklabels(fairness_df['Race'])
ax.set_ylabel('False Positive Rate')
ax.set_title('False Positive Rate by Race: Neural Network vs COMPAS')
ax.legend()
plt.tight_layout()
plt.show()

---

## Question #1: False Positives and False Negatives in Recidivism Prediction (3 marks)

Answer the following question based on your results from Tasks #5–#8. Write your answer in the markdown cell below.

Look at the classification report from logistic regression (Task #5). In the context of **recidivism prediction**, consider false positives (predicting recidivism when it will not happen) and false negatives (predicting no recidivism when it will happen).

Your answer should address **all three** of the following for full marks:

1. **Define** both error types in the context of this problem — what does each mean concretely for a defendant? (1 mark)
2. **Identify who bears the cost** of each error type — who is harmed by a false positive and who is harmed by a false negative? (1 mark)
3. **Take a position** on which error is more costly in this setting and explain your reasoning. There is no single correct answer, but your argument should be clearly connected to the stakes involved. (1 mark)

---

Place your answers for Question #1 in this cell

---

---

## Question #2: Ethics — Algorithmic Fairness (3 marks)

**Before answering, re-read the [ProPublica article](https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing) and [methodology](https://www.propublica.org/article/how-we-analyzed-the-compas-recidivism-algorithm).**

ProPublica argued that COMPAS is racially biased because Black defendants have a higher false positive rate than White defendants. Northpointe (the company that made COMPAS) responded that the algorithm is fair because it has equal **predictive accuracy** across racial groups (i.e., a score of 7 predicts similar recidivism rates for both groups).

### Background: The Impossibility of Simultaneous Fairness

Two important papers, published independently in 2017, proved that ProPublica and Northpointe cannot both be satisfied at the same time — not because of a flaw in COMPAS, but because of a **mathematical impossibility**:

- **Chouldechova (2017)**, [*Fair Prediction with Disparate Impact: A Study of Bias in Recidivism Prediction Instruments*](https://doi.org/10.1089/big.2016.0047) — proved that when the base rate of the outcome (here, recidivism) differs between groups, it is impossible for a risk score to simultaneously have equal **false positive rates** and equal **positive predictive values** across groups. In other words, if Black and White defendants recidivate at different rates, no algorithm — no matter how well designed — can be "fair" by both ProPublica's and Northpointe's definitions at the same time.

- **Kleinberg, Mullainathan, and Raghavan (2017)**, [*Inherent Trade-Offs in the Fair Determination of Risk Scores*](https://doi.org/10.4230/LIPIcs.ITCS.2017.43) — proved a more general version of the same result: three natural fairness conditions (calibration, balance for the positive class, and balance for the negative class) cannot all hold simultaneously unless the base rates are equal across groups or the predictor is perfect. Since neither condition holds in practice, trade-offs are unavoidable.

### Your answer should address **all three** of the following for full marks:

1. **Explain each side's fairness criterion** — what specific metric is ProPublica using, and what specific metric is Northpointe using? (1 mark)
2. **Can both claims be true at the same time?** Explain why or why not, drawing on the impossibility results described above. (1 mark)
3. **What does this tell you about the difficulty of defining algorithmic fairness?** Discuss what this disagreement implies for the use of algorithms in high-stakes decisions. (1 mark)

---

Place your answer for Question #2 in this cell

---

---

## Question #3: Ethics — Should Algorithms Be Used in Criminal Justice? (3 marks)

There is no uniquely correct answer — we are looking for thoughtful engagement with the trade-offs.

Your models achieve roughly 65–70% accuracy on predicting two-year recidivism. Should algorithmic tools like COMPAS (or the models you built) be used to inform bail, sentencing, or parole decisions?

Your answer should address **all three** of the following for full marks:

1. **Take a clear position** — should these tools be used, not used, or used only under certain conditions? Justify your position with reference to the accuracy numbers and the stakes of the decision. (1 mark)
2. **Identify specific conditions or safeguards** — what would need to be true about accuracy, fairness, or the decision-making process for you to be comfortable (or more comfortable) with such a tool being used? Give at least two concrete conditions. (1 mark)
3. **Acknowledge a counterargument** — identify the strongest argument against your position and explain why you find your own position more compelling despite it. (1 mark)

---

Place your answers for Question #3 in this cell

---